# Gold Layer Aggregation

This notebook reads cleansed Silver Delta tables and produces analytical Gold tables
optimized for Power BI Direct Lake consumption. It creates:

- **gold_cost_summary** — Daily/monthly cost breakdowns with YTD running totals
- **gold_capacity_usage** — Capacity utilization metrics (avg, max, p95)
- **gold_operational_metrics** — Error rates, latency percentiles, availability
- **gold_resource_inventory** — Enriched resource list with SCD Type 2 history

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import (
    StructType, StructField, StringType, DateType,
    DecimalType, IntegerType, TimestampType, BooleanType, DoubleType
)
from datetime import date

## Gold Cost Summary

Aggregates daily and monthly costs by service, capacity, and region. A window function
computes year-to-date (YTD) running totals partitioned by year and ordered by month,
using `rangeBetween(Window.unboundedPreceding, Window.currentRow)`.

In [ ]:
silver_costs = spark.read.format("delta").load("Tables/silver_costs")

daily_costs = (
    silver_costs
    .withColumn("cost_date", F.col("cost_date").cast(DateType()))
    .withColumn("cost_amount", F.col("cost_amount").cast(DecimalType(18, 4)))
    .groupBy("cost_date", "service_name", "capacity_name", "region")
    .agg(
        F.sum("cost_amount").alias("daily_cost"),
        F.count("*").alias("line_item_count")
    )
)

monthly_costs = (
    daily_costs
    .withColumn("cost_year", F.year("cost_date"))
    .withColumn("cost_month", F.month("cost_date"))
    .groupBy("cost_year", "cost_month", "service_name", "capacity_name", "region")
    .agg(
        F.sum("daily_cost").alias("monthly_cost"),
        F.sum("line_item_count").alias("monthly_line_items"),
        F.min("cost_date").alias("period_start"),
        F.max("cost_date").alias("period_end")
    )
)

ytd_window = (
    Window
    .partitionBy("cost_year", "service_name", "capacity_name", "region")
    .orderBy("cost_month")
    .rangeBetween(Window.unboundedPreceding, Window.currentRow)
)

gold_cost_summary = (
    monthly_costs
    .withColumn("ytd_cost", F.sum("monthly_cost").over(ytd_window))
    .withColumn("ytd_line_items", F.sum("monthly_line_items").over(ytd_window))
    .withColumn("period_start", F.col("period_start").cast(DateType()))
    .withColumn("period_end", F.col("period_end").cast(DateType()))
    .withColumn("monthly_cost", F.col("monthly_cost").cast(DecimalType(18, 4)))
    .withColumn("ytd_cost", F.col("ytd_cost").cast(DecimalType(18, 4)))
)

(
    gold_cost_summary
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .save("Tables/gold_cost_summary")
)

print(f"gold_cost_summary written — {gold_cost_summary.count()} rows")

## Gold Capacity Usage

Computes per-workspace / per-capacity utilization metrics including average, maximum,
and 95th-percentile utilization using `percentile_approx`.

In [ ]:
silver_capacity = spark.read.format("delta").load("Tables/silver_capacity")

gold_capacity_usage = (
    silver_capacity
    .withColumn("utilization_pct", F.col("utilization_pct").cast(DoubleType()))
    .withColumn("snapshot_date", F.col("snapshot_date").cast(DateType()))
    .groupBy("workspace_id", "workspace_name", "capacity_id", "capacity_name")
    .agg(
        F.avg("utilization_pct").cast(DecimalType(10, 4)).alias("avg_utilization_pct"),
        F.max("utilization_pct").cast(DecimalType(10, 4)).alias("max_utilization_pct"),
        F.expr("percentile_approx(utilization_pct, 0.95)").cast(DecimalType(10, 4)).alias("p95_utilization_pct"),
        F.min("snapshot_date").alias("earliest_snapshot"),
        F.max("snapshot_date").alias("latest_snapshot"),
        F.count("*").alias("sample_count")
    )
    .withColumn("earliest_snapshot", F.col("earliest_snapshot").cast(DateType()))
    .withColumn("latest_snapshot", F.col("latest_snapshot").cast(DateType()))
)

(
    gold_capacity_usage
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .save("Tables/gold_capacity_usage")
)

print(f"gold_capacity_usage written — {gold_capacity_usage.count()} rows")

## Gold Operational Metrics

Derives per-service operational health indicators:
- **Error rate** — ratio of error-level events to total events
- **Latency percentiles** — p50, p90, p99 via `percentile_approx`
- **Availability** — percentage of time intervals with at least one successful response

In [ ]:
silver_operations = spark.read.format("delta").load("Tables/silver_operations")

gold_operational_metrics = (
    silver_operations
    .withColumn("is_error", F.when(F.col("severity") == "Error", 1).otherwise(0))
    .withColumn("is_success", F.when(F.col("status_code").between(200, 299), 1).otherwise(0))
    .withColumn("latency_ms", F.col("latency_ms").cast(DoubleType()))
    .withColumn("metric_date", F.col("event_timestamp").cast(DateType()))
    .groupBy("service_name", "metric_date")
    .agg(
        F.count("*").alias("total_events"),
        F.sum("is_error").alias("error_count"),
        (F.sum("is_error") / F.count("*")).cast(DecimalType(10, 6)).alias("error_rate"),
        F.expr("percentile_approx(latency_ms, 0.50)").cast(DecimalType(10, 2)).alias("latency_p50_ms"),
        F.expr("percentile_approx(latency_ms, 0.90)").cast(DecimalType(10, 2)).alias("latency_p90_ms"),
        F.expr("percentile_approx(latency_ms, 0.99)").cast(DecimalType(10, 2)).alias("latency_p99_ms"),
        F.sum("is_success").alias("success_count"),
        (F.sum("is_success") / F.count("*") * 100).cast(DecimalType(10, 4)).alias("availability_pct")
    )
    .withColumn("metric_date", F.col("metric_date").cast(DateType()))
)

(
    gold_operational_metrics
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .save("Tables/gold_operational_metrics")
)

print(f"gold_operational_metrics written — {gold_operational_metrics.count()} rows")

## Gold Resource Inventory

Joins Silver resources with Silver costs to build an enriched resource catalogue.
Compliance flags are derived from tag presence and naming conventions.
This table uses **SCD Type 2** logic so historical resource state is preserved.

In [ ]:
silver_resources = spark.read.format("delta").load("Tables/silver_resources")
silver_costs_for_join = spark.read.format("delta").load("Tables/silver_costs")

resource_cost_agg = (
    silver_costs_for_join
    .groupBy("resource_id")
    .agg(
        F.sum("cost_amount").cast(DecimalType(18, 4)).alias("total_cost"),
        F.avg("cost_amount").cast(DecimalType(18, 4)).alias("avg_daily_cost"),
        F.max("cost_date").cast(DateType()).alias("latest_cost_date")
    )
)

new_inventory = (
    silver_resources
    .join(resource_cost_agg, on="resource_id", how="left")
    .withColumn("has_environment_tag",
        F.when(F.col("tags").contains("environment"), True).otherwise(False)
    )
    .withColumn("has_owner_tag",
        F.when(F.col("tags").contains("owner"), True).otherwise(False)
    )
    .withColumn("has_cost_center_tag",
        F.when(F.col("tags").contains("costCenter"), True).otherwise(False)
    )
    .withColumn("naming_compliant",
        F.col("resource_name").rlike("^[a-z][a-z0-9\\-]{2,62}$")
    )
    .withColumn("compliance_score",
        (
            F.col("has_environment_tag").cast(IntegerType())
            + F.col("has_owner_tag").cast(IntegerType())
            + F.col("has_cost_center_tag").cast(IntegerType())
            + F.col("naming_compliant").cast(IntegerType())
        )
    )
    .withColumn("compliance_status",
        F.when(F.col("compliance_score") == 4, "Compliant")
         .when(F.col("compliance_score") >= 2, "Partial")
         .otherwise("Non-Compliant")
    )
    .withColumn("total_cost", F.coalesce(F.col("total_cost"), F.lit(0).cast(DecimalType(18, 4))))
    .withColumn("avg_daily_cost", F.coalesce(F.col("avg_daily_cost"), F.lit(0).cast(DecimalType(18, 4))))
    .withColumn("snapshot_date", F.current_date())
    .select(
        F.col("resource_id").cast(StringType()),
        F.col("resource_name").cast(StringType()),
        F.col("resource_type").cast(StringType()),
        F.col("resource_group").cast(StringType()),
        F.col("region").cast(StringType()),
        F.col("tags").cast(StringType()),
        "total_cost",
        "avg_daily_cost",
        F.col("latest_cost_date").cast(DateType()),
        "has_environment_tag",
        "has_owner_tag",
        "has_cost_center_tag",
        "naming_compliant",
        "compliance_score",
        F.col("compliance_status").cast(StringType()),
        F.col("snapshot_date").cast(DateType())
    )
)

print(f"New inventory snapshot prepared — {new_inventory.count()} resources")

### SCD Type 2 — Slowly Changing Dimension Handling

On each run the notebook compares the incoming resource snapshot with the existing
`gold_resource_inventory` table:

1. **Changed records** — `effective_end_date` is set to yesterday and `is_current` becomes `false`.
2. **New / changed records** — inserted with `effective_start_date = current_date` and `is_current = true`.
3. **Initial load** — if the table does not yet exist, all records are inserted as current.

In [ ]:
from pyspark.sql.utils import AnalysisException

scd_columns = [
    "resource_name", "resource_type", "resource_group", "region",
    "tags", "compliance_status", "total_cost"
]

new_with_scd = (
    new_inventory
    .withColumn("effective_start_date", F.current_date())
    .withColumn("effective_end_date", F.lit(None).cast(DateType()))
    .withColumn("is_current", F.lit(True))
    .withColumn("record_hash",
        F.sha2(F.concat_ws("|", *[F.coalesce(F.col(c).cast(StringType()), F.lit("")) for c in scd_columns]), 256)
    )
)

table_exists = False
try:
    existing = spark.read.format("delta").load("Tables/gold_resource_inventory")
    table_exists = existing.count() > 0
except AnalysisException:
    table_exists = False

if not table_exists:
    (
        new_with_scd
        .write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .save("Tables/gold_resource_inventory")
    )
    print("Initial load — gold_resource_inventory created")
else:
    existing_current = existing.filter(F.col("is_current") == True)
    existing_historical = existing.filter(F.col("is_current") == False)

    existing_hashed = existing_current.withColumn("existing_hash",
        F.sha2(F.concat_ws("|", *[F.coalesce(F.col(c).cast(StringType()), F.lit("")) for c in scd_columns]), 256)
    )

    changed = (
        existing_hashed
        .join(new_with_scd.select("resource_id", "record_hash"), on="resource_id", how="inner")
        .filter(F.col("existing_hash") != F.col("record_hash"))
    )

    closed_records = (
        changed
        .withColumn("effective_end_date", F.date_sub(F.current_date(), 1))
        .withColumn("is_current", F.lit(False))
        .drop("existing_hash", "record_hash")
    )

    changed_ids = [row.resource_id for row in changed.select("resource_id").collect()]

    unchanged_current = existing_current.filter(~F.col("resource_id").isin(changed_ids))

    new_records = new_with_scd.filter(
        F.col("resource_id").isin(changed_ids)
        | ~F.col("resource_id").isin(
            [row.resource_id for row in existing_current.select("resource_id").collect()]
        )
    ).drop("record_hash")

    final = (
        existing_historical
        .unionByName(closed_records, allowMissingColumns=True)
        .unionByName(unchanged_current, allowMissingColumns=True)
        .unionByName(new_records, allowMissingColumns=True)
    )

    (
        final
        .write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .save("Tables/gold_resource_inventory")
    )

    print(f"SCD2 merge complete — {len(changed_ids)} changed, {new_records.count()} new/changed inserted")

## Power BI Direct Lake Views

Register temporary Spark SQL views so that downstream notebooks or a Lakehouse
SQL analytics endpoint can expose them to Power BI Direct Lake mode.
All columns use explicit Power BI-friendly data types.

In [ ]:
spark.read.format("delta").load("Tables/gold_cost_summary").createOrReplaceTempView("vw_gold_cost_summary")
spark.read.format("delta").load("Tables/gold_capacity_usage").createOrReplaceTempView("vw_gold_capacity_usage")
spark.read.format("delta").load("Tables/gold_operational_metrics").createOrReplaceTempView("vw_gold_operational_metrics")
spark.read.format("delta").load("Tables/gold_resource_inventory").createOrReplaceTempView("vw_gold_resource_inventory")

for view_name in [
    "vw_gold_cost_summary",
    "vw_gold_capacity_usage",
    "vw_gold_operational_metrics",
    "vw_gold_resource_inventory"
]:
    row_count = spark.sql(f"SELECT COUNT(*) AS cnt FROM {view_name}").collect()[0]["cnt"]
    print(f"{view_name}: {row_count} rows")

print("\nGold layer aggregation complete.")